In [96]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [97]:
import os
os.getcwd()

'/content'

In [98]:
import os

os.makedirs("data", exist_ok=True)
os.makedirs("model", exist_ok=True)

In [99]:
df = pd.read_csv("data/data_ikm_clean_final.csv")
df.shape

(17697, 9)

In [101]:
from sklearn.tree import DecisionTreeClassifier

# retrain model FINAL
# Using reasonable default parameters for max_depth and min_samples_leaf
# as 'best_dt' is not defined in the current context.
# Assuming X_full corresponds to X_final and y_full corresponds to y (target variable).
model_final = DecisionTreeClassifier(
    max_depth=10, # Example value, adjust as needed
    min_samples_leaf=5, # Example value, adjust as needed
    random_state=42
)

model_final.fit(X_final, y)

# TIMPA model lama
joblib.dump(model_final, "model/decision_tree.pkl")

['model/decision_tree.pkl']

In [102]:
model = joblib.load("model/decision_tree.pkl")
encoder = joblib.load("model/encoder.pkl")

In [103]:
cat_cols = [
    "Uraian Jenis Perusahaan",
    "Uraian Risiko Proyek",
    "Judul Kbli",
    "kecamatan_usaha"
]

num_cols = [
    "Jumlah Investasi",
    "luas_tanah",
    "TKI"
]

In [104]:
X_cat = encoder.transform(df[cat_cols])
X_num = df[num_cols].values
X_final = np.hstack([X_num, X_cat])

df["Skala Usaha Prediksi"] = model.predict(X_final)

In [105]:
df = df.reset_index(drop=True)

X_cat = encoder.transform(df[cat_cols])
X_num = df[num_cols].values
X_final = np.hstack([X_num, X_cat])

print("df:", df.shape)
print("X_cat:", X_cat.shape)
print("X_num:", X_num.shape)
print("X_final:", X_final.shape)
print("pred:", model.predict(X_final).shape)


df: (17697, 10)
X_cat: (17697, 496)
X_num: (17697, 3)
X_final: (17697, 499)
pred: (17697,)


In [106]:
df[cat_cols].isnull().sum()

,0
Uraian Jenis Perusahaan,0
Uraian Risiko Proyek,0
Judul Kbli,0
kecamatan_usaha,0


In [112]:
hasil_klasifikasi = df[[
    "Nama Perusahaan",
    "kecamatan_usaha",
    "Judul Kbli",
    "Uraian Jenis Perusahaan",
    "Uraian Risiko Proyek",
    "Jumlah Investasi",
    "TKI",
    "luas_tanah",
    "Uraian Skala Usaha",
    "Skala Usaha Prediksi"
]].copy()

hasil_klasifikasi.head()

,Nama Perusahaan,kecamatan_usaha,Judul Kbli,Uraian Jenis Perusahaan,Uraian Risiko Proyek,Jumlah Investasi,TKI,luas_tanah,Uraian Skala Usaha,Skala Usaha Prediksi
0,MEDIKA ANUGERAH LANGGENG,Gunung Anyar,"Industri Kosmetik Untuk Manusia, Termasuk Past...",PT,Rendah,200000000,2,10.0,Usaha Mikro,Usaha Kecil
1,WIRAMULIA KRIDASATRYA,Tambaksari,Industri Furnitur Dari Kayu,PT,Rendah,400000000,7,1.0,Usaha Mikro,Usaha Mikro
2,WIRAMULIA KRIDASATRYA,Tambaksari,Industri Barang Bangunan Dari Kayu,PT,Rendah,400000000,2,1.0,Usaha Mikro,Usaha Mikro
3,KARYA SUMBER DJATI,Tandes,Industri Bumbu Rokok Serta Kelengkapan Rokok L...,PT,Tinggi,1070000000,8,200.0,Usaha Kecil,Usaha Kecil
4,KALIMAYA GLOBAL PERSADA,Sawahan,Reparasi Mobil,PT,Menengah Rendah,50000000,2,50.0,Usaha Mikro,Usaha Mikro


In [111]:
prioritas_pembinaan = hasil_klasifikasi[
    (hasil_klasifikasi["Uraian Skala Usaha"] == "Usaha Mikro") &
    (hasil_klasifikasi["Skala Usaha Prediksi"] != "Usaha Mikro")
].copy()


prioritas_pembinaan.shape

(253, 10)

In [113]:
prioritas_kecamatan = (
    prioritas_pembinaan
    .groupby("kecamatan_usaha")
    .size()
    .reset_index(name="Jumlah Usaha Prioritas")
    .sort_values(by="Jumlah Usaha Prioritas", ascending=False)
)


prioritas_kecamatan.head()

,kecamatan_usaha,Jumlah Usaha Prioritas
13,Mulyorejo,27
17,Sambikerep,25
16,Rungkut,24
7,Gunung Anyar,23
6,Gubeng,14


In [114]:
hasil_klasifikasi.to_excel("data/hasil_klasifikasi_skala_usaha_lengkap.xlsx", index=False)
prioritas_pembinaan.to_excel("data/prioritas_pembinaan_ikm_lengkap.xlsx", index=False)
prioritas_kecamatan.to_excel("data/prioritas_kecamatan_pembinaan.xlsx", index=False)